<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student_attempt3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

I will start off by looking at the data sample to understand the data more. I just want to look at the first few rows from each table

prompt : filter first 5 rows from each table and provide the total number of row from each table

In [4]:
for name, df in tables.items():
    print(f"Table: {name} ({df.shape[0]:,} rows)")
    display(df.head(5))
    print("-" * 50)

Table: trips (612 rows)


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


--------------------------------------------------
Table: bikes (60 rows)


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022


--------------------------------------------------
Table: stations (12 rows)


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0


--------------------------------------------------


Okay straight away i can see some formating issue with column user_type from table trip and column bike_type from table bike so i will ensure to do the data checking and cleaning throughly

I will start off by checking for any missing value from the 3 tables. Hopefully there are none

prompt : check for any missing value from the 3 tables. the output should seperate by table and by column

In [5]:
for name, df in tables.items():
    print(f"=== Missing values in '{name}' table ===")
    missing_counts = df.isnull().sum()
    display(missing_counts[missing_counts >= 0])
    print("\n")

=== Missing values in 'trips' table ===


,0
trip_id,0
bike_id,0
start_station_id,0
end_station_id,0
start_time,0
duration_min,30
user_type,0




=== Missing values in 'bikes' table ===


,0
bike_id,0
bike_type,0
purchase_year,0




=== Missing values in 'stations' table ===


,0
station_id,0
station_name,0
zone,0
capacity,1


There are 30 missing value in column duration_min from table trips and 1 missing_value from column capacity from table stations

let filter out these data to investigate more on

prompt : filter out records with missing value

In [6]:
# Filter and display records with missing 'duration_min' in the trips table
trips_missing = tables['trips'][tables['trips']['duration_min'].isnull()]
print(f"Missing duration in trips: {len(trips_missing)} rows")
display(trips_missing.head())

# Filter and display records with missing 'capacity' in the stations table
stations_missing = tables['stations'][tables['stations']['capacity'].isnull()]
print(f"\nMissing capacity in stations: {len(stations_missing)} rows")
display(stations_missing)

Missing duration in trips: 30 rows


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
12,T0013,B024,S04,S02,01/03/2025 18:43,NaN,Member
14,T0015,B040,S03,S04,2025-03-01 19:03:00,NaN,Member
30,T0031,B015,S06,S06,2025-03-02 08:50:00,NaN,Casual
60,T0061,B011,S09,S07,2025-03-04 08:23:00,NaN,Member
64,T0065,B041,S06,S02,04/03/2025 09:06,NaN,Member



Missing capacity in stations: 1 rows


,station_id,station_name,zone,capacity
8,S09,Old Town Hall,Centre,NaN


Okay I think I will keep these records and decide later on what i should do with them

I will move on to check the data duplication from the 3 tables. Hopefully there is no duplication in any of the table

prompt : check for any exact duplication across 3 tables. The output should be shown by table and if there are any duplication filter those record out

In [7]:
for name, df in list(tables.items()):
    # Only check the original tables
    if name in ['trips', 'bikes', 'stations']:
        duplicates_count = df.duplicated().sum()
        print(f"Table '{name}': {duplicates_count} duplicate rows found.")
        if duplicates_count > 0:
            # Filter out duplicates
            tables[name] = df.drop_duplicates()
            print(f"  -> Removed {duplicates_count} duplicates. New row count: {len(tables[name])}")

Table 'trips': 12 duplicate rows found.
  -> Removed 12 duplicates. New row count: 600
Table 'bikes': 0 duplicate rows found.
Table 'stations': 0 duplicate rows found.


okay AI remove the duplication right away once it found the duplicate row

let me redo this

prompt : I want to check for any duplication from the original 3 tables and if there are any print it out for me to verify it and do not take any action yet

In [8]:
# Load fresh copies of original tables to check duplicates before any removal
import glob
import os
import pandas as pd

for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, '**', filename), recursive=True)
    if found:
        original_df = pd.read_csv(found[0])
        dupes = original_df[original_df.duplicated(keep=False)]
        print(f"=== Table '{name}': {original_df.duplicated().sum()} duplicate rows found ===")
        if not dupes.empty:
            display(dupes.sort_values(by=list(original_df.columns)).head(10))
        else:
            print("No duplicate records found.")
        print("\n")

=== Table 'trips': 12 duplicate rows found ===


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member




=== Table 'bikes': 0 duplicate rows found ===
No duplicate records found.


=== Table 'stations': 0 duplicate rows found ===
No duplicate records found.




okay nice this is the result that i was expacting. So there are 12 exact duplication in the table trips. I will take note of this for now and process it later with other actions

next i will check for all the primary keys from each table whether it's unique so for table trip i expact the number of unique value for trip_id to be the same as number of row of the trip value. for table bike i expact the number of unique bike_id to be the same as number of row in table bike and i expact the number of unique station_id to be the same as number of row from table station

prompt : check the number of unique value of trip_id from table trip compare with number of row from table trip, bike_id from table bike compare with number of row from table bike, and station_id from table station compare with number of row from table station

In [9]:
import glob
import os
import pandas as pd

# Load fresh copies of original tables to make a clean comparison
original_tables = {}
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, '**', filename), recursive=True)
    if found:
        original_tables[name] = pd.read_csv(found[0])

# Compare unique keys vs total row counts
for name, key in [('trips', 'trip_id'), ('bikes', 'bike_id'), ('stations', 'station_id')]:
    df = original_tables[name]
    total_rows = len(df)
    unique_keys = df[key].nunique()
    print(f"=== Table '{name}' ===")
    print(f"Total rows: {total_rows}")
    print(f"Unique {key} values: {unique_keys}")
    if total_rows == unique_keys:
        print(f"✓ SUCCESS: {key} is unique and can serve as a primary key.")
    else:
        print(f"✗ WARNING: {key} is NOT unique! (Difference: {total_rows - unique_keys})")
    print()


=== Table 'trips' ===
Total rows: 612
Unique trip_id values: 600
✗ WARNING: trip_id is NOT unique! (Difference: 12)

=== Table 'bikes' ===
Total rows: 60
Unique bike_id values: 60
✓ SUCCESS: bike_id is unique and can serve as a primary key.

=== Table 'stations' ===
Total rows: 12
Unique station_id values: 12
✓ SUCCESS: station_id is unique and can serve as a primary key.



okay so the number of unique value of primary key match with the number of row for table bikes and stations while for table trips there are 12 duplications which match with the number of exact duplicates recorrds we found earlier

Next I will check the formating of all the column with list as a value to check for the use of uppercase/lowercase, and white space whether it consistent

prompt : check for all the column on the formating by printing out unique value from column user_type, bike_type, station_name and zone

In [10]:
# Retrieve tables
trips_df = original_tables['trips']
bikes_df = original_tables['bikes']
stations_df = original_tables['stations']

# Print unique values for each specified column to check formatting
print("=== Unique values in trips.user_type ===")
print(trips_df['user_type'].unique())
print()

print("=== Unique values in bikes.bike_type ===")
print(bikes_df['bike_type'].unique())
print()

print("=== Unique values in stations.station_name ===")
print(stations_df['station_name'].unique())
print()

print("=== Unique values in stations.zone ===")
print(stations_df['zone'].unique())
print()

=== Unique values in trips.user_type ===
['Casual' 'CASUAL' 'Member' 'MEMBER' 'Casual ' 'member' 'casual']

=== Unique values in bikes.bike_type ===
['Standard' 'Electric ' 'electric' 'STANDARD' 'standard' 'Electric'
 'E-bike']

=== Unique values in stations.station_name ===
['Market Square' 'Railway Station' 'Canal Basin' 'University Gate'
 'Park Road' 'Library Corner' 'Hospital Entrance' 'Riverside Walk'
 'Old Town Hall' 'Retail Park' 'Sports Centre' 'Cathedral Close']

=== Unique values in stations.zone ===
['Centre' 'North' 'West' 'South' 'East']



okay there are some formatting issue for column user_type and bike_type while station_name and zone look consistent

I will check for the formating of the datetime column as well whether it use consistent formating across the 3 tables and within the same table

prompt : check for the datetime formating use across the 3 tables whether it consistent within the same table and also across the 3 table

In [11]:
# Retrieve the trips dataframe
trips_df = original_tables['trips']

# Let's inspect a few values to see if there is any visible format mix
print("=== Sample start_time values ===")
print(trips_df['start_time'].head(15))
print()

# Let's test how pandas parses them and if any fail
try:
    parsed_dates = pd.to_datetime(trips_df['start_time'], errors='raise')
    print("✓ SUCCESS: All datetime values are parseable into timestamps.")
except Exception as e:
    print(f"✗ ERROR: Datetime parsing failed. Details: {e}")

# Let's check for inconsistent formats (e.g., matching standard YYYY-MM-DD vs DD/MM/YYYY)
has_dash = trips_df['start_time'].str.contains('-').sum()
has_slash = trips_df['start_time'].str.contains('/').sum()
print(f"\nNumber of rows with '-' format: {has_dash}")
print(f"Number of rows with '/' format: {has_slash}")


=== Sample start_time values ===
0     2025-03-01 05:52:00
1     2025-03-01 06:31:00
2     2025-03-01 07:06:00
3     2025-03-01 07:06:00
4     2025-03-01 07:36:00
5     2025-03-01 08:06:00
6        01/03/2025 09:26
7     2025-03-01 10:35:00
8     2025-03-01 12:14:00
9     2025-03-01 13:04:00
10    2025-03-01 13:05:00
11    2025-03-01 13:09:00
12       01/03/2025 18:43
13    2025-03-01 19:00:00
14    2025-03-01 19:03:00
Name: start_time, dtype: object

✗ ERROR: Datetime parsing failed. Details: time data "01/03/2025 09:26" doesn't match format "%Y-%m-%d %H:%M:%S", at position 6. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

Number of rows with '-' format: 514
Number of rows with '/' format: 9

okay there are some mix formating in column start_time which will need to be clean but i will finish all the checking first before i start on the cleaning

I want to check the formating of the ID columns as well on the formating whether it consistent within the table and across so when we join the data there are no issue

prompt : check the formating for column trip_id, bike_id, start_station_id, end_station_id, station_id. I expact all the bike_id to match in the formating and i also expact start_station_id, end_station_id and station_id to match

In [12]:
# Retrieve the dataframes
trips_df = original_tables['trips']
bikes_df = original_tables['bikes']
stations_df = original_tables['stations']

# Helper function to check format characteristics of a column
def check_id_format(df, col_name):
    # Drop nulls for checking formats
    series = df[col_name].dropna().astype(str)
    lengths = series.str.len().unique()
    has_spaces = series.str.contains(' ').any()
    # Check prefix pattern (e.g., starts with 'T', 'B', 'S')
    prefixes = series.str[0].unique()
    print(f"Column '{col_name}': lengths={list(lengths)}, unique prefixes={list(prefixes)}, contains spaces={has_spaces}")

print("=== General Format Characteristics ===")
check_id_format(trips_df, 'trip_id')
check_id_format(trips_df, 'bike_id')
check_id_format(bikes_df, 'bike_id')
check_id_format(trips_df, 'start_station_id')
check_id_format(trips_df, 'end_station_id')
check_id_format(stations_df, 'station_id')
print()

# Verification of ID matches across tables
print("=== Key Integrity Verification ===")
# 1. Check if all bike_id values in trips exist in bikes table
trips_bikes = set(trips_df['bike_id'].dropna().unique())
bikes_master = set(bikes_df['bike_id'].unique())
unmatched_bikes = trips_bikes - bikes_master
print(f"Bike IDs in trips but not in bikes table: {unmatched_bikes if unmatched_bikes else 'None (100% Match)'}")

# 2. Check if all start_station_id and end_station_id values in trips exist in stations table
trips_start_stations = set(trips_df['start_station_id'].dropna().unique())
trips_end_stations = set(trips_df['end_station_id'].dropna().unique())
stations_master = set(stations_df['station_id'].unique())

unmatched_start = trips_start_stations - stations_master
unmatched_end = trips_end_stations - stations_master

print(f"Start Station IDs in trips but not in stations table: {unmatched_start if unmatched_start else 'None (100% Match)'}")
print(f"End Station IDs in trips but not in stations table: {unmatched_end if unmatched_end else 'None (100% Match)'}")

=== General Format Characteristics ===
Column 'trip_id': lengths=[np.int64(5)], unique prefixes=['T'], contains spaces=False
Column 'bike_id': lengths=[np.int64(4)], unique prefixes=['B'], contains spaces=False
Column 'bike_id': lengths=[np.int64(4)], unique prefixes=['B'], contains spaces=False
Column 'start_station_id': lengths=[np.int64(3)], unique prefixes=['S'], contains spaces=False
Column 'end_station_id': lengths=[np.int64(3)], unique prefixes=['S'], contains spaces=False
Column 'station_id': lengths=[np.int64(3)], unique prefixes=['S'], contains spaces=False

=== Key Integrity Verification ===
Bike IDs in trips but not in bikes table: None (100% Match)
Start Station IDs in trips but not in stations table: None (100% Match)
End Station IDs in trips but not in stations table: None (100% Match)


okay all match so when we do the data joining using primary key and foreign key there shoundn't any issue that cause by data not matching in format

i forgot to check the formating of the year so will quickly do that

prompt : check the formating of column year whether it match with the format of year use in the start_time

In [13]:
# Retrieve the datasets
bikes_df = original_tables['bikes']
trips_df = original_tables['trips']

# Check unique values in purchase_year
print("=== Unique Values in bikes.purchase_year ===")
print(sorted(bikes_df['purchase_year'].unique()))
print(f"Data type of purchase_year: {bikes_df['purchase_year'].dtype}")
print()

# Extract unique years from trips.start_time
# Since some dates are YYYY-MM-DD and others are DD/MM/YYYY, we extract the 4-digit year using regex
years_in_start_time = trips_df['start_time'].astype(str).str.extract(r'(\b20\d{2}\b)')[0].unique()
print("=== Unique Years Extracted from trips.start_time ===")
print(sorted(years_in_start_time))


=== Unique Values in bikes.purchase_year ===
[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Data type of purchase_year: int64

=== Unique Years Extracted from trips.start_time ===
['2025']


okay they both use the same so will lastly check the statistic for duramin_min and capacity to identify any outlier

prompt : check the statistic of duration_min and capacity. I want to see min, max, sd, 25%, and 75%

In [14]:
import pandas as pd

# Retrieve dataframes from the original_tables dict
trips_df = original_tables['trips']
stations_df = original_tables['stations']

print("=== Descriptive Statistics: trips.duration_min ===")
display(trips_df['duration_min'].describe(percentiles=[0.25, 0.75]))
print("\n")

print("=== Descriptive Statistics: stations.capacity ===")
display(stations_df['capacity'].describe(percentiles=[0.25, 0.75]))

=== Descriptive Statistics: trips.duration_min ===


,duration_min
count,582.000000
mean,27.891753
std,145.064305
min,4.000000
25%,12.000000
50%,16.000000
75%,22.000000
max,2880.000000




=== Descriptive Statistics: stations.capacity ===


,capacity
count,11.000000
mean,22.909091
std,5.539938
min,14.000000
25%,19.000000
50%,24.000000
75%,27.500000
max,29.000000


the max value for duration_min definitely seems to indicate that there is an outlier which i will tackel in the next section

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

prompt : summarize all the finding we found from the looking at data section

### Summary of Diagnostic Findings

During our initial data exploration, we identified the following issues that must be addressed before joining and aggregating:

1. **Duplicates:**
   - **`trips` Table:** Contains **12 exact duplicate rows** (6 pairs of identical trips). These must be removed to prevent double-counting.
   - **`bikes` & `stations` Tables:** Clean with 0 duplicates.

2. **Missing Values:**
   - **`trips.duration_min`:** Contains **30 missing values**.
   - **`stations.capacity`:** Contains **1 missing value** for `'Old Town Hall'` (Station `S09`).

3. **Text Casing & Spacing Inconsistencies:**
   - **`trips.user_type`:** Has mixed values due to case changes and trailing whitespaces (e.g., `'Casual'`, `'CASUAL'`, `'Casual '`, `'Member'`, `'MEMBER'`, `'member'`, `'casual'`). These must be normalized to standard clean values: `'Casual'` and `'Member'`.
   - **`bikes.bike_type`:** Similar inconsistency (e.g., `'Standard'`, `'STANDARD'`, `'standard'`, `'Electric'`, `'electric'`, `'Electric '`, and `'E-bike'`). These must be mapped consistently to `'Standard'` and `'Electric'`.

4. **Datetime Formatting (Mixed formats):**
   - **`trips.start_time`:** Uses two distinct styles: 514 rows use the standard `YYYY-MM-DD HH:MM:SS` format, and 98 rows use the British/European slash format `DD/MM/YYYY HH:MM`. Standardizing these with `pd.to_datetime(..., format='mixed', dayfirst=True)` is required.

5. **Keys & Logical Alignment:**
   - Primary keys (`bike_id`, `station_id`) are unique once duplicates are cleared from `trips`.
   - All ID columns have 100% referential integrity (no missing links or rogue IDs during joins).
   - `bikes.purchase_year` (2021-2024) logically precedes all recorded trip times (all occurring in 2025).

6. **Outliers:**
   - **`trips.duration_min`:** Includes an extreme maximum of **2,880.0 minutes (48 hours)** compared to a 75th percentile of just 22.0 minutes. This outlier needs to be handled to avoid distorting averages.

okay so based on our findings I will fix the duplication since it will impact when we do data joining. I will leave the data missing but will make sure when we do calculation like average to remove the NA value. I will fix the text casing and whitespace as well as the datetime format. Also will be removing some of the extreme outlier so it won't bias the result

first will start by cleaning the trips table. since im a bit short on time i will combine all the action at once

prompt : remove the duplication for trips table as well as fixing the text casing and whitespace for user_type. I want to format for user_type to start with capital letter and the rest is lowercase with no whitespace in front and after the word. And fix the formating for start time to be year-month-day hour:min:sec for example 2026-07-05 07:35:20. And duration min to be remove if the value exceed 600. Please create all this as new table

*i use 600 for duration due to the limitation of time. we should properly check for outlier before deciding the cut of value

In [15]:
import pandas as pd

# Retrieve a copy of the original trips table
trips_clean = original_tables['trips'].copy()

# 1. Remove exact duplicate rows
trips_clean = trips_clean.drop_duplicates()
print(f"Rows after removing duplicates: {len(trips_clean)}")

# 2. Trim whitespace and format user_type to Title Case
trips_clean['user_type'] = trips_clean['user_type'].astype(str).str.strip().str.title()
print("Unique user_types after cleaning:", trips_clean['user_type'].unique())

# 3. Parse and standardize start_time to 'YYYY-MM-DD HH:MM:SS'
trips_clean['start_time'] = pd.to_datetime(trips_clean['start_time'], format='mixed', dayfirst=True)
trips_clean['start_time'] = trips_clean['start_time'].dt.strftime('%Y-%m-%d %H:%M:%S')
print("Sample standardized start times:")
print(trips_clean['start_time'].head())

# 4. Filter out records where duration_min > 600 (keeping NaN values)
trips_clean = trips_clean[(trips_clean['duration_min'] <= 600) | (trips_clean['duration_min'].isna())]
print(f"Rows remaining after removing durations > 600: {len(trips_clean)}")

# Save the cleaned table in our active tables dictionary
tables['trips_clean'] = trips_clean

Rows after removing duplicates: 600
Unique user_types after cleaning: ['Casual' 'Member']
Sample standardized start times:
0    2025-03-01 05:52:00
1    2025-03-01 06:31:00
2    2025-03-01 07:06:00
3    2025-03-01 07:06:00
4    2025-03-01 07:36:00
Name: start_time, dtype: object
Rows remaining after removing durations > 600: 597


okay number of row after remove duplication does match with the unique value of trip_id. and the formating of user_type and start_time look okay now so will quickly move on to the next table

prompt : fixing the formating for bike_type to start with capital letter and the rest is lowercase with no whitespace in front and after the word and create all this into new table

In [16]:
import pandas as pd

# Retrieve a copy of the original bikes table
bikes_clean = original_tables['bikes'].copy()

# 1. Strip whitespace and convert to Title Case
bikes_clean['bike_type'] = bikes_clean['bike_type'].astype(str).str.strip().str.title()

# 2. Standardize 'E-Bike' mapping to 'Electric'
bikes_clean['bike_type'] = bikes_clean['bike_type'].replace({'E-Bike': 'Electric'})

print("Unique bike types after cleaning:", bikes_clean['bike_type'].unique())
print(f"Total unique bikes: {len(bikes_clean)}")

# Save the cleaned table in our active tables dictionary as 'bikes_clean'
tables['bikes_clean'] = bikes_clean

Unique bike types after cleaning: ['Standard' 'Electric']
Total unique bikes: 60


the unique value for bike_type look nice now but will quickly check the table again by ensuring that the number of row still match the orginal table

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

do not have time for join and aggregate

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

already add check throughout the steps

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

already add prompt throughout the steps

## 6. What I could not verify

*What remains unchecked? What would you do next?*

do not have time to find the exact value to
determine outlier

do not have time to do proper check for data cleaning

do not have time to do aggregation in the end for final table

